In [1]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from eeg.eeg_data import EEGLLM, EEGDataset

/root/eeg/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
eeg_dataset: EEGDataset = EEGDataset(region_tokenizer_path="../models/appendages", print_shapes=True)

Initializing dataset...
Getting VQVAE model...
Getting EEG data...
Filtered & processed EEG data.
Getting appendage data...
Retrieved appendage data.
Successful retrieved all data.
Pre-computing VQ-VAE tokens...


100%|██████████| 25/25 [01:00<00:00,  2.42s/it]


In [4]:
def load_model(epoch: int) -> EEGLLM:
    model = EEGLLM(
        vocab_size=512,
        num_layers=4,
        num_heads=4,
        num_channels=14,
        embedding_dim=64,
        ffn_hidden_dim=64,
        qk_length=64,
        value_length=64,
        max_length=900,
        dropout=0.1
    )

    state_dict = torch.load(f"/var/log/thavamount/eeg_ckpts/eeg_lm/big/eeg_llm_epch_{epoch}.pth", map_location="cuda")
    model.load_state_dict(state_dict["model"])
    
    return model

In [ ]:
eeg, apps, tokens = hand_dataset_cnn.get_validation_data(0)

In [ ]:
def inference(model: torch.nn.Module, seen_so_far: int = 100) -> torch.Tensor:
    model.eval()
    all_region_tokens = region_tokens.unsqueeze(0).to(torch.int64)

    first_region_token = all_region_tokens[:, :seen_so_far]

    region_tokens_so_far = list(first_region_token.tolist()[0])
    appendage_values_so_far = [torch.tensor(appendage_values[:seen_so_far])]

    for _ in tqdm(range(200)):
        region_token_logits, pred_appendage = model(
            torch.tensor(region_tokens_so_far).unsqueeze(0)
        )

        best_region_token = torch.argmax(region_token_logits.squeeze(0)[-1])
        region_tokens_so_far.append(best_region_token.item())

        # region_probs = torch.softmax(region_token_logits, dim=1).squeeze(0)[-1]
        # print(region_probs.shape)
        # best_region_token = torch.multinomial(region_probs, num_samples=1).item()
        # print(best_region_token)

        last_pred_appendage = pred_appendage.squeeze(0)[-1]  # (12,)
        appendage_values_so_far.append(last_pred_appendage)
    
    appendage_values_without_first = torch.stack(appendage_values_so_far[1:]).detach()
    appendage_values_so_far = torch.cat(
        [torch.tensor(appendage_values[:seen_so_far]), appendage_values_without_first], dim=0
    ).detach()

    return region_tokens_so_far, appendage_values_so_far  # (T, 12)